# Spine XR — Project 3 FINALE (Advisor brief: Traditional vs ProGAN vs Diffusion)

Colab Pro+ (A100). ROI-128 patch strategy kept. **Backbone: DenseNet-121.** **3-fold CV + held-out test.** Online train-only augmentation. Generative arms: **MONAI DDPM** + **ProGAN** (WGAN retired).

Karşılaştırma: `baseline / traditional / diffusion / progan` — hepsi DenseNet-121, 3-fold CV, resmi test'te macro-F1 (mean±std) + sentetik kalite (FID).

**Dürüstlük:** ROI 'lokalizasyon verili' bir görevdir (NF yamaları boyut-eşli, R3). Baseline ~90-95% olduğu için headroom az; katkı = kontrollü karşılaştırma + FID.

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr-augmentation-study/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

Yeni bağımlılıklar: monai, monai-generative, pytorch-fid (requirements.txt). MONAI generative monai≥1.4 çekirdeğinde de olabilir; import sarmalayıcı iki yolu da dener.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!python -c "import monai; print('monai', monai.__version__)"

## 2. Audit → Splits → ROI patches → CV folds

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!python scripts/02b_build_roi_patches.py --config configs/base.yaml --cases configs/cases.yaml
!python scripts/02c_make_roi_folds.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/02b_roi/summary.md

## 3. CV Baseline — DenseNet-121 (no aug)
Beklenti: test macro-F1 mean±std ≈ önceki tek-split 0.90–0.95.

In [ ]:
!python scripts/03_train_cv.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform baseline_no_aug \
    --out-tag 03cv_baseline
!cat outputs/03cv_baseline/cv_summary.md

## 4. CV Traditional — online aug + balanced sampler

In [ ]:
!python scripts/03_train_cv.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform roi_train_online --balanced-sampler \
    --out-tag 04cv_traditional
!cat outputs/04cv_traditional/cv_summary.md

## 5. Diffusion arm — MONAI pixel-space DDPM (128²)

### 5.0 Smoke (Other lesions, 300 iter) — hata vermeden çalışmalı, sample lezyon dokusu

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Other lesions" --iterations 300 --out-tag 05d_smoke
from pathlib import Path; import matplotlib.pyplot as plt, cv2
s=sorted(Path('outputs/05d_smoke/case_4/other_lesions/samples').glob('*.png'))
if s: plt.figure(figsize=(6,6)); plt.imshow(cv2.imread(str(s[-1]),0),cmap='gray'); plt.axis('off'); plt.show()

### 5.1 DDPM — Disc space narrowing

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Disc space narrowing" --out-tag 05d_diffusion

### 5.2 DDPM — Vertebral collapse

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Vertebral collapse" --out-tag 05d_diffusion

### 5.3 DDPM — Foraminal stenosis

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Foraminal stenosis" --out-tag 05d_diffusion

### 5.4 DDPM — Spondylolysthesis

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Spondylolysthesis" --out-tag 05d_diffusion

### 5.5 DDPM — Surgical implant

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Surgical implant" --out-tag 05d_diffusion

### 5.6 DDPM — Other lesions

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Other lesions" --out-tag 05d_diffusion

### 5.gen Üret (her sınıf, son checkpoint) → manifest

In [ ]:
import subprocess
from pathlib import Path
dmap=[('case_1','disc_space_narrowing'),('case_1','vertebral_collapse'),('case_2','foraminal_stenosis'),
      ('case_2','spondylolysthesis'),('case_3','surgical_implant'),('case_4','other_lesions')]
for case,slug in dmap:
    cks=sorted(Path(f'outputs/05d_diffusion/{case}/{slug}/checkpoints').glob('*.pt'))
    if not cks: print('NO CKPT',slug); continue
    subprocess.run(['python','scripts/07d_generate_diffusion.py','--checkpoint',str(cks[-1]),
                    '--n-samples','800','--batch-size','64'],check=True)

### 5.fid Sentetik kalite (FID, gerçek patch'lere karşı)

In [ ]:
import sys; sys.path.insert(0,'.')
import pandas as pd
from src.eval.fid import compute_fid, real_paths_for_class
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),
      ('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),
      ('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
rows=[]
for nm,case,slug in dmap:
    real=real_paths_for_class(f'outputs/02b_roi/{case}/train.csv', nm)
    man=f'outputs/07d_diffusion_generated/{slug}/manifest.csv'
    fake=pd.read_csv(man)['path'].tolist()
    try: fid=compute_fid(real,fake,device='cuda')
    except Exception as e: fid=float('nan'); print(slug,'FID err',e)
    rows.append({'class':nm,'n_real':len(real),'n_fake':len(fake),'FID':round(fid,2)})
print(pd.DataFrame(rows).to_string(index=False))

### 5.cv CV Diffusion-augmented (sentetik patch'ler train fold'larına eklenir)

In [ ]:
GLOB='outputs/07d_diffusion_generated'
import glob
mans=sorted(glob.glob(f'{GLOB}/*/manifest.csv'))
print('manifests:',mans)
import subprocess
cmd=['python','scripts/03_train_cv.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
     '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
     '--out-tag','05cv_diffusion','--synth-manifest']+mans
subprocess.run(cmd,check=True)
print(open('outputs/05cv_diffusion/cv_summary.md').read())

## 6. ProGAN arm (en riskli — diffusion çalıştıktan sonra)

### 6.0 Smoke (Other lesions, kısa fade/stab)

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Other lesions" --fade-iters 300 --stab-iters 300 --out-tag 05p_smoke
from pathlib import Path; import matplotlib.pyplot as plt, cv2
s=sorted(Path('outputs/05p_smoke/case_4/other_lesions/samples').glob('*.png'))
if s: plt.figure(figsize=(6,6)); plt.imshow(cv2.imread(str(s[-1]),0),cmap='gray'); plt.axis('off'); plt.show()

### 6.1 ProGAN — Disc space narrowing

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Disc space narrowing" --out-tag 05p_progan

### 6.2 ProGAN — Vertebral collapse

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Vertebral collapse" --out-tag 05p_progan

### 6.3 ProGAN — Foraminal stenosis

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Foraminal stenosis" --out-tag 05p_progan

### 6.4 ProGAN — Spondylolysthesis

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Spondylolysthesis" --out-tag 05p_progan

### 6.5 ProGAN — Surgical implant

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Surgical implant" --out-tag 05p_progan

### 6.6 ProGAN — Other lesions

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Other lesions" --out-tag 05p_progan

### 6.gen Üret (her sınıf, depth5 checkpoint) → manifest

In [ ]:
import subprocess
from pathlib import Path
for case,slug in dmap if False else [('case_1','disc_space_narrowing'),('case_1','vertebral_collapse'),
     ('case_2','foraminal_stenosis'),('case_2','spondylolysthesis'),('case_3','surgical_implant'),('case_4','other_lesions')]:
    ck=Path(f'outputs/05p_progan/{case}/{slug}/checkpoints/depth5_res128.pt')
    if not ck.exists(): print('NO CKPT',slug); continue
    subprocess.run(['python','scripts/07p_generate_progan.py','--checkpoint',str(ck),
                    '--n-samples','800','--batch-size','64'],check=True)

### 6.cv CV ProGAN-augmented

In [ ]:
import glob, subprocess
mans=sorted(glob.glob('outputs/07p_progan_generated/*/manifest.csv'))
print('manifests:',mans)
cmd=['python','scripts/03_train_cv.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
     '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
     '--out-tag','06cv_progan','--synth-manifest']+mans
subprocess.run(cmd,check=True)
print(open('outputs/06cv_progan/cv_summary.md').read())

## 7. Final karşılaştırma tablosu — tüm koşullar

In [ ]:
import json, pandas as pd
from pathlib import Path
conds=[('03cv_baseline','baseline'),('04cv_traditional','traditional'),
       ('05cv_diffusion','diffusion'),('06cv_progan','progan')]
recs=[]
for tag,lbl in conds:
    f=Path(f'outputs/{tag}/cv_summary.json')
    if not f.exists(): continue
    for s in json.loads(f.read_text()):
        recs.append({'case':s['case'],'condition':lbl,
                     'macroF1':f"{s['test_macro_f1_mean']:.4f}±{s['test_macro_f1_std']:.4f}"})
df=pd.DataFrame(recs)
if len(df): print(df.pivot(index='case',columns='condition',values='macroF1').to_string())